# Lab 7: Model Selection, Inference, and Transformations

This lab uses the `Hitters` dataset (1986/87 MLB player salaries and performance). A cleaned copy is included with the lab as `hitters.csv`. There are three parts:

* Part 1 quantifies how badly the "select-then-infer" workflow from Lecture #13 inflates the Type I error rate on real `Hitters` covariates.
* Part 2 walks through the proper primary/secondary workflow with `Years` as the pre-specified covariate of interest.
* Part 3 covers transformations: when `Salary` is right-skewed, fitting `log(Salary) ~ Years` buys us statistical power that `Salary ~ Years` doesn't.

If you are running the lab on DataHub, keep `hitters.csv` in the same folder as this notebook. You do not need to install any extra R packages for the dataset.

In [1]:
# Please run this initialization cell
library(ottr)

data_path <- "hitters.csv"
if (!file.exists(data_path) && file.exists("/autograder/source/hitters.csv")) {
    data_path <- "/autograder/source/hitters.csv"
}
if (!file.exists(data_path)) {
    stop("Could not find hitters.csv. Make sure it is in the same folder as this notebook.")
}

hitters <- read.csv(data_path, stringsAsFactors = TRUE)
hitters$League <- factor(hitters$League)
hitters$Division <- factor(hitters$Division)
hitters$NewLeague <- factor(hitters$NewLeague)
cat("Rows:", nrow(hitters), "\n")
head(hitters)

`hitters` has 263 rows (the original 322 player-seasons, minus the ones with missing salary). The columns we will use:

| Column | Description |
|--------|-------------|
| `Salary` | 1987 annual salary (in thousands of dollars) |
| `Years` | Number of years in the major leagues |
| `Hits` | Number of hits in 1986 |
| `HmRun` | Number of home runs in 1986 |
| `CHmRun` | Number of home runs over the player's career |
| `Runs` | Number of runs in 1986 |
| `RBI` | Number of runs batted in in 1986 |
| `Walks` | Number of walks in 1986 |
| `CAtBat` | Number of times at bat over career |
| `PutOuts` | Number of put outs in 1986 |
| `League` | League (A or N) at the end of 1986 |
| `Division` | Division (E or W) at the end of 1986 |

The response is `Salary`. Our primary covariate of interest is `Years`.

## Part 1: Type I Error inflation from data-dependent model selection

Lecture #13 showed that the following workflow is not valid:

> Fit the full model. Drop a secondary covariate if its $p$-value is large. Report the $p$-value on the primary covariate from whichever model survives.

When two covariates are correlated, deciding whether to keep one of them based on its in-sample $p$-value leaks information into the slope of the other. The reported $p$-value no longer has the long-run rejection rate it claims.

In this part we hold a pair of `Hitters` covariates fixed and simulate a fake response under $H_0$ (drawn with no relationship to anything). Since we know there is no effect, every rejection is a Type I error, which lets us count them directly.

Pairs of covariates in `Hitters` span a wide range of correlation strengths. Cumulative career stats like `CAtBat` (career at-bats) build up over time, so they are nearly perfectly correlated with `Years`. Single-season counts like `Hits` (hits in 1986 alone) are not. The cell below prints three correlations that we will use as a low / medium / high collinearity ladder in the questions that follow:

In [2]:
cat("cor(Years, Hits)   =", round(cor(hitters$Years, hitters$Hits),   3), "\n")
cat("cor(Years, CHmRun) =", round(cor(hitters$Years, hitters$CHmRun), 3), "\n")
cat("cor(Years, CAtBat) =", round(cor(hitters$Years, hitters$CAtBat), 3), "\n")

**Question 1.1.** A sports analytics team wants to test whether `Years` predicts `Salary`. They are deciding between three two-covariate models, and in each case they plan to drop the second covariate if its $p$-value exceeds 0.05 and report the `Years` $p$-value from whichever model survives:

* Model A: `Salary ~ Years + Hits` (`cor(Years, Hits)` $\approx 0.02$)
* Model B: `Salary ~ Years + CHmRun` (`cor(Years, CHmRun)` $\approx 0.72$)
* Model C: `Salary ~ Years + CAtBat` (`cor(Years, CAtBat)` $\approx 0.92$)

Under which specification is the long-run Type I error rate on the reported `Years` $p$-value most distorted? Assign `q1_1_ans` to one of `1`, `2`, `3`, or `4`.

1. Model A. Nearly uncorrelated covariates create the largest distortion after selection.
2. Model B. The middle correlation setting is worst, while low and high correlations are safer.
3. Model C. When `CAtBat` is dropped, its highly correlated variation can leak into `Years` the most.
4. None of the three. The workflow remains valid whenever the dropped covariate has no true salary effect.

<!--
BEGIN QUESTION
name: q1_1
-->


In [ ]:
q1_1_ans <- ...

In [ ]:
. = ottr::check("tests/q1_1.R")

**Question 1.2.** Write a function `sim_select_then_infer(reps, x1, x2)` that takes two numeric covariate vectors of equal length `n` and runs the select-then-infer simulation under $H_0$. For each replicate:

1. Draw a fresh response `y <- rnorm(n)`. Since `y` has no relationship to `x1` or `x2`, every rejection is a Type I error.
2. Fit `lm(y ~ x1 + x2)`. Record the $p$-value on `x1` as the full-model $p$-value.
3. If the $p$-value on `x2` is greater than 0.05, drop `x2` and refit `lm(y ~ x1)`. Record that model's $p$-value on `x1` as the selected-model $p$-value. Otherwise, the selected-model $p$-value is the same as the full-model one.

Return a list with two numeric vectors of length `reps`: `pval_full` and `pval_selected`.

We will pass real `hitters` columns as `x1` and `x2` in Q1.3. Because they are fixed across replicates, the only randomness comes from `y`, and `cor(x1, x2)` plays the role that `rho` did in lecture.

After you define the function, call it with `reps = 200`, `x1 = hitters$Years`, and `x2 = hitters$CHmRun`, and store the result in `check_q12`. The printed rates are a quick sanity check for your simulation.

<!--
BEGIN QUESTION
name: q1_2
-->


In [ ]:
sim_select_then_infer <- function(reps, x1, x2) {
    ...
}

set.seed(414) # DO NOT DELETE OR CHANGE THIS LINE!
check_q12 <- ...
cat("full Type I:", mean(check_q12$pval_full < 0.05),
    "  selected Type I:", mean(check_q12$pval_selected < 0.05), "\n")

In [ ]:
. = ottr::check("tests/q1_2.R")

**Question 1.3.** Run `sim_select_then_infer` with `reps = 2000` on each of the three pairs from Q1.1:

* `x1 = hitters$Years`, `x2 = hitters$Hits`
* `x1 = hitters$Years`, `x2 = hitters$CHmRun`
* `x1 = hitters$Years`, `x2 = hitters$CAtBat`

For each pair, compute the empirical rejection rates at $\alpha = 0.05$ from the full-model and selected-model $p$-values. Put everything in a data frame `rates_q13` with columns `pair`, `cor_x1_x2`, `rate_full`, `rate_selected`, ordered from lowest to highest correlation (`Years/Hits` first, `Years/CAtBat` last).

The starter code runs the three simulations and builds the data frame from four intermediate vectors. Fill in the two rate vectors.

<!--
BEGIN QUESTION
name: q1_3
-->


In [ ]:
set.seed(414) # DO NOT DELETE OR CHANGE THIS LINE!

sim_low  <- sim_select_then_infer(reps = 2000, hitters$Years, hitters$Hits)
sim_med  <- sim_select_then_infer(reps = 2000, hitters$Years, hitters$CHmRun)
sim_high <- sim_select_then_infer(reps = 2000, hitters$Years, hitters$CAtBat)

pair_q13 <- c("Years/Hits", "Years/CHmRun", "Years/CAtBat")
cor_q13 <- c(cor(hitters$Years, hitters$Hits),
             cor(hitters$Years, hitters$CHmRun),
             cor(hitters$Years, hitters$CAtBat))
rate_full_q13 <- ...
rate_selected_q13 <- ...

rates_q13 <- data.frame(pair = pair_q13,
                        cor_x1_x2 = cor_q13,
                        rate_full = rate_full_q13,
                        rate_selected = rate_selected_q13,
                        stringsAsFactors = FALSE)

rates_q13

In [ ]:
. = ottr::check("tests/q1_3.R")

The chart below compares the full-model rejection rate (always valid under $H_0$) against the select-then-infer rate for each pair.

In [9]:
mat_q13 <- rbind(full = rates_q13$rate_full,
                 select_then_infer = rates_q13$rate_selected)
colnames(mat_q13) <- sprintf("%s\n(r=%.2f)", rates_q13$pair, rates_q13$cor_x1_x2)

bp <- barplot(mat_q13, beside = TRUE,
              col = c("steelblue", "firebrick"),
              ylim = c(0, max(0.15, max(mat_q13) * 1.2)),
              ylab = "Empirical Type I error (alpha = 0.05)",
              main = "Full model vs. select-then-infer on Hitters pairs")
abline(h = 0.05, lty = 2)
legend("topleft", legend = c("Full model", "Select-then-infer model"),
       fill = c("steelblue", "firebrick"), bty = "n")

The full-model bars sit near the dashed 0.05 line across all three pairs, which is what we want under $H_0$. The select-then-infer bars tell a different story: at the `Years/Hits` pair (almost no correlation) the workflow looks fine, at the `Years/CHmRun` pair (moderate correlation) it has drifted noticeably above nominal, and at the `Years/CAtBat` pair (strong correlation) it's clearly inflated. The takeaway is that select-then-infer isn't always a disaster, but the moment two covariates are meaningfully correlated, the reported $p$-value stops doing what it claims.

**Question 1.4.** Which statement best describes the pattern in `rates_q13` and the bar chart? Assign `q1_4_ans` to one of `1`, `2`, or `3`.

1. The full-model rates stay near 5%, while the select-then-infer rates rise as the covariates become more correlated.
2. Both workflows drift upward together, so the inflation is caused by the `Hitters` covariates rather than selection.
3. The select-then-infer rates stay at or below 5%, so dropping noisy covariates restores valid inference.

<!--
BEGIN QUESTION
name: q1_4
-->


In [ ]:
q1_4_ans <- ...

In [ ]:
. = ottr::check("tests/q1_4.R")

**Question 1.5.** For the `Years/CAtBat` pair, by approximately what factor does the select-then-infer workflow inflate the nominal 5% rate? Compute the ratio from `rates_q13` (the selected-model rate divided by 0.05) and store it in `inflation_factor_q15`.

Interpretation: a value of 1 would mean no inflation, while a value of 1.6 would mean the workflow rejects about 60% more often than the advertised 5% rate.

<!--
BEGIN QUESTION
name: q1_5
-->


In [ ]:
inflation_factor_q15 <- ...
inflation_factor_q15

In [ ]:
. = ottr::check("tests/q1_5.R")

## Part 2: The proper primary/secondary workflow on `Hitters`

The proper alternative to select-then-infer has two stages:

1. A primary analysis on a single, pre-specified model. The $p$-value reported here has the nominal Type I error rate.
2. Secondary analyses (diagnostics, sensitivity checks, permutation tests). These are useful for checking robustness and informing follow-up studies, but they do not replace the primary analysis.

Suppose the scientific question is: *holding 1986 performance and league/division fixed, do veteran players earn more than rookies?* Then a reasonable pre-specified model is
$$
\texttt{Salary} \sim \texttt{Years} + \texttt{Hits} + \texttt{HmRun} + \texttt{Runs} + \texttt{RBI} + \texttt{Walks} + \texttt{PutOuts} + \texttt{League} + \texttt{Division},
$$
with `Years` as the primary covariate of interest.

**Question 2.1.** Fit the pre-specified primary model. Store the fit in `primary_fit`, the $p$-value on `Years` in `pval_primary`, and a logical `reject_primary` that is `TRUE` iff `pval_primary < 0.05` (i.e., we reject $H_0: \beta_{\texttt{Years}} = 0$ at $\alpha = 0.05$).

<!--
BEGIN QUESTION
name: q2_1
-->


In [ ]:
primary_fit <- ...
pval_primary <- ...
reject_primary <- ...

pval_primary
reject_primary

In [ ]:
. = ottr::check("tests/q2_1.R")

The primary analysis is now finished. We reject $H_0: \beta_{\texttt{Years}} = 0$ and report a positive association between major-league experience and salary, holding 1986 performance and league/division fixed. That conclusion is the only thing we will publish from this dataset about the `Years`–`Salary` relationship; everything that follows is secondary.

### Sensitivity analysis: backward elimination

As discussed in Lecture #13, a sensitivity analysis asks whether the primary conclusion is robust to reasonable analysis choices. Here the analysis choice is how many secondary covariates to keep. We will use backward elimination as a secondary check, not as a replacement for the pre-specified model above.

The procedure is: fit the model, find the non-`Years` covariate with the largest $p$-value, drop it if that $p$-value is above 0.05, refit, repeat until everything left has $p < 0.05$. To get you started, here's round 1 worked out on the primary model:

In [16]:
round1_pvals <- summary(primary_fit)$coefficients[, "Pr(>|t|)"]
print(round(round1_pvals, 4))  # which non-Years covariate has the biggest p-value?

`RBI` has the largest $p$-value (~0.88), so we drop it and refit:

In [17]:
fit_step1 <- lm(Salary ~ Years + Hits + HmRun + Runs + Walks + PutOuts + League + Division,
                data = hitters)
step1_pvals <- summary(fit_step1)$coefficients[, "Pr(>|t|)"]
print(round(step1_pvals, 4))

Now `Runs` has the largest $p$-value. From here, you take over.

**Question 2.2 (sensitivity analysis).** Continue the backward elimination from the cell above. At each step, drop the non-`Years` covariate with the largest $p$-value (provided that $p$-value is above 0.05) and refit. Keep going until every remaining non-`Years` covariate has $p < 0.05$. (For two-level factors like `League` and `Division`, use the single dummy's $p$-value.)

When you stop, store the final model in `final_fit`, the `Years` $p$-value from it in `pval_sensitivity`, and a character vector of the covariates that survived (excluding `"(Intercept)"`, in any order) in `final_model_vars`. Use the original covariate names in `final_model_vars`; for example, write `Division`, not the dummy coefficient name `DivisionW`.

<!--
BEGIN QUESTION
name: q2_2
-->


In [ ]:
...

pval_sensitivity
final_model_vars

In [ ]:
. = ottr::check("tests/q2_2.R")

Your sensitivity-analysis $p$-value for `Years` should be smaller than the primary-analysis $p$-value. It is tempting to say *"my smaller model gives a more significant result, I'll report that one instead."* The next two questions are about why that temptation should be resisted: first the mechanism of the fallacy, then the appropriate role of a sensitivity analysis.

**Question 2.3.** In Part 1 we saw that dropping *one* secondary covariate based on its $p$-value inflated the Type I error rate on the primary covariate, with the inflation growing with the correlation between the two. The sensitivity analysis in Q2.2 is structurally the same workflow, just iterated across multiple covariates: at every step we used the data to decide which covariate to drop next. Why does this iterated process make `pval_sensitivity` an unreliable $p$-value for `Years`? Assign `q2_3_ans` to one of `1`, `2`, `3`, or `4`.

1. Dropping covariates mainly changes the degrees of freedom, so the final $t$-statistic is calibrated by the wrong table.
2. The same data chose `final_fit` and then supplied its `Years` $p$-value, so the usual 0.05 calibration is too optimistic.
3. Backward elimination is a heuristic, so the selected coefficients lose their numerical meaning after the model changes.
4. Removing variables always makes the `Years` standard error smaller, even when `Years` is truly unrelated to salary.

<!--
BEGIN QUESTION
name: q2_3
-->


In [ ]:
q2_3_ans <- ...

In [ ]:
. = ottr::check("tests/q2_3.R")

**Question 2.4.** What is the legitimate role of the sensitivity analysis from Q2.2? Assign `q2_4_ans` to one of `1`, `2`, `3`, or `4`.

1. To replace the primary analysis whenever the reduced model gives a clearer story.
2. To search for the smallest `Years` $p$-value and report that model as the main result.
3. To check robustness and guide future modeling choices while leaving the primary result as the reportable test.
4. To show that any covariate removed by backward elimination should never have been included.

<!--
BEGIN QUESTION
name: q2_4
-->


In [ ]:
q2_4_ans <- ...

In [ ]:
. = ottr::check("tests/q2_4.R")

## Part 3: Transformations

Lecture #14 introduced transformations as a way to recover the LINE conditions when the relationship between $y$ and $x$ is non-linear. Before we get into that, take a look at the marginal distribution of `Salary` against `log(Salary)`:

In [24]:
par(mfrow = c(1, 2))
hist(hitters$Salary, breaks = 30, col = "steelblue",
     xlab = "Salary (thousands $)", main = "Salary")
hist(log(hitters$Salary), breaks = 30, col = "darkorange",
     xlab = "log(Salary)", main = "log(Salary)")
par(mfrow = c(1, 1))

`Salary` is right-skewed, while `log(Salary)` is roughly symmetric. This already suggests that `log(Salary) ~ Years` is a more reasonable functional form for the salary–experience relationship than `Salary ~ Years`. The questions below quantify the power gain from fitting the correctly transformed model.

**Question 3.1.** Fit two simple linear regressions on the `hitters` data and extract the $p$-values for the slope on `Years`:

* Store the model using untransformed `Salary` as the response in `fit_untransformed`, and its `Years` $p$-value in `pval_untransformed`.
* Store the model using `log(Salary)` as the response in `fit_logged`, and its `Years` $p$-value in `pval_logged`.

<!--
BEGIN QUESTION
name: q3_1
-->


In [ ]:
fit_untransformed <- ...
fit_logged        <- ...

pval_untransformed <- ...
pval_logged        <- ...

cat("pval_untransformed =", pval_untransformed, "\n")
cat("pval_logged        =", pval_logged, "\n")

In [ ]:
. = ottr::check("tests/q3_1.R")

Now we will run a power simulation in which the truth is linear on the log scale. To make the simulated data look like real `Hitters`, we use the intercept, slope, and residual SD from the real `lm(log(Salary) ~ Years)` fit as the ground-truth parameters.

In [27]:
beta0_truth <- coef(fit_logged)[1]
beta1_truth <- coef(fit_logged)[2]
sigma_truth <- summary(fit_logged)$sigma

cat("beta0 (intercept on log scale)  =", round(beta0_truth, 3), "\n")
cat("beta1 (slope on log scale)      =", round(beta1_truth, 3), "\n")
cat("sigma (residual SD on log scale)=", round(sigma_truth, 3), "\n")

**Question 3.2.** Write a function `sim_power_logy(reps, n, beta1, sigma)` that simulates data where the truth is linear on the log scale and compares the power of the two candidate models. For each replicate:

1. Draw `years` of length `n` from $\text{Uniform}(1, 24)$.
2. Simulate `log_salary` $= \beta_0 + \beta_1 \cdot \texttt{years} + \varepsilon$ with $\varepsilon \sim N(0, \sigma)$, using `beta0_truth` from the cell above as $\beta_0$.
3. Convert to `salary` $= \exp(\texttt{log\_salary})$.
4. Fit `lm(salary ~ years)` and `lm(log_salary ~ years)` and check whether each rejects $H_0: \beta_1 = 0$ at $\alpha = 0.05$.

Return a named list with two scalars: `power_untransformed` and `power_logged`.

Your function can refer to `beta0_truth` directly via R's lexical scoping.

<!--
BEGIN QUESTION
name: q3_2
-->


In [ ]:
sim_power_logy <- function(reps, n, beta1, sigma) {
    ...
}

In [ ]:
. = ottr::check("tests/q3_2.R")

**Question 3.3.** Run `sim_power_logy(reps = 1000, n = 50, beta1 = beta1_truth / 4, sigma = sigma_truth)`. The quarter-strength slope and reduced sample size keep both models below saturation. Put the result in a one-row data frame `power_q33` with columns `power_untransformed` and `power_logged`.

<!--
BEGIN QUESTION
name: q3_3
-->


In [ ]:
set.seed(414) # DO NOT DELETE OR CHANGE THIS LINE!

...

power_q33

In [ ]:
. = ottr::check("tests/q3_3.R")

The two powers in `power_q33` should be roughly 0.25 and 0.32. Both models are fitting data where the truth is linear on the log scale, but the untransformed model is forced to fit a straight line through an exponential mean function. That misspecification inflates the residuals and shrinks the $t$-statistic on the slope, which is exactly the ~7-percentage-point gap you see. The untransformed model is correct on average (the slope is unbiased in expectation), but you reject $H_0$ less often when you do have a real effect to detect.

**Question 3.4.** Let $\hat{\beta}_1$ be the slope on `Years` in `lm(log(Salary) ~ Years)`. What is its interpretation on the `Salary` scale? Assign `q3_4_ans` to one of `1`, `2`, `3`, or `4`.

1. Each extra year adds about $\hat{\beta}_1$ thousand dollars to `Salary` on the original scale.
2. Each extra year multiplies typical `Salary` by about $\exp(\hat{\beta}_1)$; subtract 1 for the percent change.
3. Each extra year adds $\log(\hat{\beta}_1)$ thousand dollars because the response was logged.
4. The slope has no salary-scale interpretation once the model is fit on the log scale.

<!--
BEGIN QUESTION
name: q3_4
-->


In [ ]:
q3_4_ans <- ...

In [ ]:
. = ottr::check("tests/q3_4.R")

**Question 3.5.** Which model has higher empirical power in `power_q33`, and why? Assign `q3_5_ans` to one of `1`, `2`, `3`, or `4`.

1. The untransformed model. Salary dollars are the scientific scale of interest.
2. The logged model. It matches the simulated mean structure and leaves less residual noise.
3. They have the same power. Both regressions use the same `Years` values and sample size.
4. The logged model. Log transformations automatically make every $p$-value smaller.

<!--
BEGIN QUESTION
name: q3_5
-->


In [ ]:
q3_5_ans <- ...

In [ ]:
. = ottr::check("tests/q3_5.R")

---

## Congratulations, you are finished!!

To submit your assignment:

1. Select `Kernel -> Restart Kernel and Run All Cells...` to ensure that you have executed all cells, including the test cells.
2. Read through the notebook to make sure everything is fine and all tests passed.
3. Download your notebook using `File -> Download`, then upload your notebook to Gradescope.
4. Stick around while the Gradescope autograder grades your work. Make sure you see that all tests have passed on Gradescope.
5. Check that you have a confirmation email from Gradescope and save it as proof of your submission.